# Aula 3, Encoder e decoder

Notebook executável que acompanha a aula [03-encoder-decoder.md](../../lessons/modulo-06-transformers/03-encoder-decoder.md).

## O que vamos fazer aqui

Vamos montar um bloco de encoder completo do zero: codificador de posição, atenção, conexão
residual, normalização e rede feed-forward. Ver o bloco inteiro funcionando mostra que um
Transformer é a repetição disciplinada de peças simples. Só numpy.

## As peças do bloco

Definimos o codificador de posição com senos e cossenos, a normalização de camada, a atenção
e a rede feed-forward, e as combinamos em um bloco de encoder com duas conexões residuais.

In [ ]:
import numpy as np

def softmax(z, eixo=-1):
    z = z - z.max(axis=eixo, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=eixo, keepdims=True)


def positional_encoding(seq_len, d):
    """Assinatura de posição com senos e cossenos de frequências variadas."""
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d)[None, :]
    ang = pos / np.power(10000, (2 * (i // 2)) / d)
    pe = np.zeros((seq_len, d))
    pe[:, 0::2] = np.sin(ang[:, 0::2])
    pe[:, 1::2] = np.cos(ang[:, 1::2])
    return pe


def layernorm(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)


def atencao(X, Wq, Wk, Wv):
    d = X.shape[1]
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    A = softmax(Q @ K.T / np.sqrt(d), eixo=-1)
    return A @ V


def encoder_block(X, Wq, Wk, Wv, W1, W2):
    att = atencao(X, Wq, Wk, Wv)
    x = layernorm(X + att)                  # residual + normalização
    ff = np.maximum(0, x @ W1) @ W2          # feed-forward com ReLU
    return layernorm(x + ff)                 # residual + normalização

## Montando o bloco sobre a frase

Somamos o codificador de posição aos embeddings e passamos pelo bloco. A saída tem a mesma
dimensão da entrada, pronta para alimentar o próximo bloco.

In [ ]:
E = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.9, 0.1, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
])

rng = np.random.default_rng(0)
pe = positional_encoding(4, 4)
X = E + pe
Wq = rng.normal(0, 1, (4, 4)); Wk = rng.normal(0, 1, (4, 4)); Wv = rng.normal(0, 1, (4, 4))
W1 = rng.normal(0, 0.5, (4, 8)); W2 = rng.normal(0, 0.5, (8, 4))

saida = encoder_block(X, Wq, Wk, Wv, W1, W2)
print("Codificação de posição da primeira posição:", np.round(pe[0], 2))
print("Saída do bloco de encoder, shape:", saida.shape)

A primeira posição recebe a assinatura `[0, 1, 0, 1]` (seno de zero é zero, cosseno
de zero é um) e o bloco devolve uma saída com a dimensão da entrada. Empilhando vários blocos,
temos um encoder, base do BERT; trocando a atenção plena pela mascarada, temos um decoder, base
do GPT. Para o projeto, empilhe dois blocos de encoder.